# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huydang2006/flyrank-ML-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Paper source:** `docs/flyrank-seo-research-march-2026.pdf`. I use the paper's SEO findings as research context and connect them to my repository validation; the paper findings are not treated as labels for this model.

**Finding 1:** The paper's position/CTR finding motivates checking whether position-tier context helps identify pages for review. In my repository, the label is a future-window CTR shortfall relative to a feature-window tier benchmark, so the result supports ranking evidence, not a causal claim about position.

**Finding 2:** The paper's impression/CTR finding motivates using minimum impression support before interpreting CTR. In my repository, impressions are used as the denominator and eligibility rule, while the label still comes from the future window. A grouped and time-aware validation design is needed before claiming that the pattern generalizes across clients or periods.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The ML-08 before result uses Jan 1–Mar 31, 2026 features and Apr 1–Apr 30, 2026 labels. It trains on 33 clients and tests on 9 completely held-out clients; the model P@50 is 0.94 and the baseline P@50 is 0.72.

ML-09 repeats the grouped split with five seeds and adds a separate time-aware design. In each time test, the model trains on one earlier 90-day feature window plus its following monthly label window, then tests on the next 90-day feature window plus its following monthly label window. The model and baseline are evaluated on all eligible rows in that test period, not only on 9 held-out clients.

Grouped results are more consistent for the model: Logistic Regression P@50 ranges from 0.88 to 0.96 across seeds, while the baseline ranges from 0.26 to 0.72. Seed 42 exactly matches ML-08 because it uses the same client split and final Jan–Mar → April period. Time-aware model P@50 ranges from 0.72 to 0.90; it beats the baseline in all three periods, but the lower April result shows period sensitivity.

In [1]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks' and ROOT.parent.name == 'work':
    ROOT = ROOT.parent.parent

result = subprocess.run(
    [sys.executable, str(ROOT / 'work' / 'scripts' / 'ml09_validation.py')],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'ml09_validation.py failed with exit code {result.returncode}')
print('Validation audit completed.')


Wrote D:\EP16\flyrank-ML-internship\work\outputs\ml09_validation_metrics.json
Wrote D:\EP16\flyrank-ML-internship\work\outputs\ml09_validation_report.md


Validation audit completed.


In [2]:
import json
metrics = json.loads((ROOT / 'work' / 'outputs' / 'ml09_validation_metrics.json').read_text())
print('Before: ML-08, Jan-Mar 2026 features -> Apr 2026 labels; 33 train clients -> 9 held-out clients.')
for row in metrics['grouped_validation']:
    print('Grouped seed', row['seed'], '| held-out clients:', row['test_clients'], '| model P@50:', round(row['model']['precision_at_50'], 4), '| baseline P@50:', round(row['baseline']['precision_at_50'], 4))
for row in metrics['time_aware_validation']:
    train_window = row['train_window']
    test_window = row['test_window']
    print('Time test | train features:', train_window[:2], '| train labels:', train_window[2:], '| test features:', test_window[:2], '| test labels:', test_window[2:], '| model P@50:', round(row['model']['precision_at_50'], 4), '| baseline P@50:', round(row['baseline']['precision_at_50'], 4))


Before: ML-08, Jan-Mar 2026 features -> Apr 2026 labels; 33 train clients -> 9 held-out clients.
Grouped seed 42 | held-out clients: 9 | model P@50: 0.94 | baseline P@50: 0.72
Grouped seed 7 | held-out clients: 9 | model P@50: 0.96 | baseline P@50: 0.26
Grouped seed 19 | held-out clients: 9 | model P@50: 0.9 | baseline P@50: 0.56
Grouped seed 31 | held-out clients: 9 | model P@50: 0.88 | baseline P@50: 0.52
Grouped seed 61 | held-out clients: 9 | model P@50: 0.92 | baseline P@50: 0.56
Time test | train features: ['2025-10-01', '2025-12-29'] | train labels: ['2026-01-01', '2026-01-31'] | test features: ['2025-11-01', '2026-01-29'] | test labels: ['2026-02-01', '2026-02-28'] | model P@50: 0.88 | baseline P@50: 0.22
Time test | train features: ['2025-11-01', '2026-01-29'] | train labels: ['2026-02-01', '2026-02-28'] | test features: ['2025-12-01', '2026-02-28'] | test labels: ['2026-03-01', '2026-03-31'] | model P@50: 0.9 | baseline P@50: 0.24
Time test | train features: ['2025-12-01', '2

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final feature list contains only feature-window numeric fields. Future label fields, pseudonymous identifiers, and duplicate client/content grains are checked explicitly.

In [3]:
for name, passed in metrics['leakage_checks'].items():
    print(f'{name}: {"PASS" if passed else "FAIL"}')
assert all(metrics['leakage_checks'].values())


duplicate_grain: PASS
feature_window_before_label_window: PASS
future_fields_excluded_from_features: PASS
identifiers_excluded_from_features: PASS


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In my evaluated FlyRank snapshot, Logistic Regression produced stronger observed ranking performance than the ML-07 rule baseline for prioritizing pages for review. 
- In the ML-08 grouped holdout, Logistic Regression reached **0.94 Precision@50** versus **0.72** for the baseline. 
- Across ML-09 grouped seeds, model Precision@50 ranged from **0.88–0.96**, versus **0.26–0.72** for the baseline; across time-aware periods, it ranged from **0.72–0.90**, versus **0.22–0.38**.

This is directional decision-support evidence, not proof that an edit will improve CTR or that the model will generalize to every future client or period.

**Limitations:**
- The rolling checks use 90-day feature windows and monthly label windows.
- Very small client groups make per-client Precision@K unstable.
- All results are directional ranking evidence, not causal effects.

In [4]:
grouped_models = [row['model']['precision_at_50'] for row in metrics['grouped_validation']]
grouped_baselines = [row['baseline']['precision_at_50'] for row in metrics['grouped_validation']]
time_models = [row['model']['precision_at_50'] for row in metrics['time_aware_validation']]
time_baselines = [row['baseline']['precision_at_50'] for row in metrics['time_aware_validation']]
print('ML-08 grouped holdout: model P@50 = 0.9400; baseline P@50 = 0.7200')
print(f'ML-09 grouped seeds: model P@50 = {min(grouped_models):.4f}-{max(grouped_models):.4f}; baseline = {min(grouped_baselines):.4f}-{max(grouped_baselines):.4f}')
print(f'ML-09 time-aware periods: model P@50 = {min(time_models):.4f}-{max(time_models):.4f}; baseline = {min(time_baselines):.4f}-{max(time_baselines):.4f}')


ML-08 grouped holdout: model P@50 = 0.9400; baseline P@50 = 0.7200
ML-09 grouped seeds: model P@50 = 0.8800-0.9600; baseline = 0.2600-0.7200
ML-09 time-aware periods: model P@50 = 0.7200-0.9000; baseline = 0.2200-0.3800


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.